In [1]:
import requests
import frontmatter

print("Setup working 🚀")

Setup working 🚀


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    prefix = "https://codeload.github.com"
    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"

    print(f"Downloading {repo_owner}/{repo_name}...")
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Download failed: {resp.status_code}")

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        if not (filename.endswith(".md") or filename.endswith(".mdx")):
            continue

        try:
            with zf.open(file_info) as f:
                content = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)

                data = post.to_dict()
                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    zf.close()
    return repository_data

In [3]:
dtc_faq = read_repo_data("DataTalksClub", "faq")

print("Total documents:", len(dtc_faq))
print(dtc_faq[0])

Total documents: 1285
{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search e

In [4]:
docs = read_repo_data("evidentlyai", "docs")

print(len(docs))

95


In [5]:
# 🚀 Pipeline Design (Day 1)

# Instead of one big function, we split into:
# 1. Download repo 📥
#    - Fetch GitHub ZIP using requests
#
# 2. Extract files 📂
#    - Use zipfile to read archive
#    - Filter .md / .mdx files
#
# 3. Parse content 🧠
#    - Use frontmatter to extract metadata + content
#
# Benefits:
# - Cleaner code
# - Easier debugging
# - Reusable components
# - Production-ready structure

In [6]:
import io
import zipfile
import requests
import frontmatter


def download_repo_zip(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to download repo: {response.status_code}")

    return io.BytesIO(response.content)


def extract_markdown_files(zip_bytes):
    zf = zipfile.ZipFile(zip_bytes)
    
    markdown_files = []

    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        if filename.endswith(".md") or filename.endswith(".mdx"):
            markdown_files.append(file_info)

    return zf, markdown_files


def parse_markdown_files(zf, markdown_files):
    data = []

    for file_info in markdown_files:
        try:
            with zf.open(file_info) as f:
                content = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)

                item = {
                    "filename": file_info.filename,
                    "metadata": post.metadata,
                    "content": post.content
                }

                data.append(item)

        except Exception as e:
            print(f"Error: {file_info.filename} -> {e}")

    return data


def load_github_repo(repo_owner, repo_name):
    zip_bytes = download_repo_zip(repo_owner, repo_name)
    zf, markdown_files = extract_markdown_files(zip_bytes)
    parsed_data = parse_markdown_files(zf, markdown_files)

    return parsed_data

In [7]:
data = load_github_repo("DataTalksClub", "faq")

print("Total files:", len(data))
print(data[0])

Total files: 1285
{'filename': 'faq-main/.claude/commands/pr.md', 'metadata': {'description': 'Review and process open FAQ PRs'}, 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomc

In [8]:
# Day 2

In [9]:
# 👉 Simple Chunking (Sliding Window)

In [10]:
evidently_docs = docs

In [11]:
print(len(evidently_docs))

95


In [12]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [13]:
doc = evidently_docs[45]

chunks = sliding_window(doc['content'], 2000, 1000)

len(chunks)

21

In [14]:
chunks[0]

{'start': 0,
 'chunk': "In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere's what we'll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.\n\n* **Build a monitoring Dashboard**. Get plo

In [15]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    chunks = sliding_window(doc_content, 2000, 1000)

    for chunk in chunks:
        chunk.update(doc_copy)

    evidently_chunks.extend(chunks)

In [16]:
len(evidently_chunks)

576

In [17]:
evidently_chunks[0]

{'start': 0,
 'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx'}

In [18]:
# 👉 Smarter chunking (paragraph + section-based)

In [19]:
import re

text = evidently_docs[45]['content']

paragraphs = re.split(r"\n\s*\n", text.strip())

len(paragraphs)

153

In [20]:
import re

def split_markdown_by_level(text, level=2):
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i+1]
        header = header.strip()

        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header

        sections.append(section)
    
    return sections

In [21]:
text = evidently_docs[45]['content']

sections = split_markdown_by_level(text, level=2)

len(sections)

8

In [22]:
sections[0]

'## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.dash

In [23]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = split_markdown_by_level(doc_content, level=2)

    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section

        evidently_chunks.append(section_doc)

In [24]:
len(evidently_chunks)

266

In [25]:
evidently_chunks[0]

{'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx',
 'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>'}

In [26]:
# 👉 Intelligent Chunking with LLM

In [35]:
from groq import Groq

client = Groq()

In [36]:
def llm(prompt, model="llama-3.1-8b-instant"):
    response = client.chat.completions.create(
        messages=[
            {"role": "user", "content": prompt}
        ],
        model=model
    )
    
    return response.choices[0].message.content

In [37]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content

---

## Another Section Name

More content

---
""".strip()

In [38]:
print("Function exists:", callable(intelligent_chunking))

Function exists: True


In [39]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    
    response = llm(prompt)
    
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    
    return sections

In [40]:
doc = evidently_docs[0]

sections = intelligent_chunking(doc['content'])

print(len(sections))

1
